# Architecture-specific control inflation: H0--H4 on case39

This notebook tests the **minimal scalar control completion** across the four physical actuator architectures H0--H3 and introduces H4 as the common full-noise oracle.

For each physical architecture,
\[
G_k=A_kR^{-1}A_k^\top,\qquad Q=\Gamma^2,
\]
and
\[
\lambda_k=\max\{\lambda:\lambda G_k\preceq Q\},\qquad
G_{+,k}=Q/\lambda_k,\qquad
\Delta G_k=G_{+,k}-G_k\succeq0.
\]
The exact completed value \(W^+_{H_k}\) is an automatic lower bound on the physical optimum for architecture \(H_k\).  The noise-deflated value \(W^-_{H_k}\) is included for comparison, but remains a candidate lower certificate until its curvature condition is verified domain-wide.

H4 is the H3-completed problem. Since H3 contains every generator-mode architecture considered here, H4 is a common lower oracle for H0--H3.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, json
ROOT=Path('..').resolve()
RES=ROOT/'results'; FIG=ROOT/'figures'
df=pd.read_csv(RES/'control_inflation_hierarchy_combined.csv')
df.head()

## Two notions of "how much control is missing"

The physical-coordinate completion burden is
\[
\mathcal I_{\rm abs}=\operatorname{tr}(G_+-G).
\]
It measures the total fictitious quadratic-control Gramian added in the chosen nodal power coordinates.

A dimensionless noise-whitened deficit is
\[
\mathcal D_Q=\operatorname{tr}\!\left[I-\lambda_{\max}Q^{-1/2}GQ^{-1/2}\right].
\]
It is invariant under a common rescaling of the noise amplitude.  With homogeneous noise and orthogonal equal-price control modes it reduces exactly to the number of unmatched stochastic directions.

These diagnostics answer different questions.  \(\mathcal D_Q\) measures **noise-direction coverage**; \(\mathcal I_{\rm abs}\) also incorporates the scalar-temperature bottleneck and is therefore more directly related to how aggressive the PIC completion must be in physical power coordinates.

In [ ]:
nom=df[df.sigma_rms.eq(0.1)][['noise','H','rank','lambda_max','virtual_rank','inflation_trace','whitened_deficit_trace','J','Wminus','Wplus','H4_oracle','gap_plus','Wplus_ess']]
nom

## Homogeneous noise: the intuitive regime

For homogeneous noise all orthonormal generator architectures have the same matching temperature
\[
\lambda_{\max}=r\sigma^2.
\]
Consequently the completed process \(G_+=Q/\lambda_{\max}\) is **the same H4 oracle for H0--H3**. Increasing physical rank simply replaces fictitious directions by real generator authority.

The dimensionless deficits are 38, 38, 36 and 29 for H0--H3.  The completed gap \(J-W^+\) decreases correspondingly.  In this isotropic setting, "missing control directions" is therefore a useful and transparent underactuation diagnostic.

## Activity-scaled noise: a scalar-matching bottleneck

The heterogeneous profile changes the picture qualitatively.  Adding an actuator direction can expose a generator with relatively weak noise. Because exact Hopf--Cole matching uses one scalar \(\lambda\), the weakest matched channel can sharply reduce \(\lambda_{\max}\). Then \(G_+=Q/\lambda_{\max}\) inflates *all* noisy directions.

At nominal noise the activity-scaled matching temperatures decrease from roughly \(5.79\times10^{-3}\) at H0 to \(4.91\times10^{-4}\) at H3. The absolute completion burden correspondingly grows from about 62 to 745 in the fixed nodal-power metric.  Thus more physical actuator rank does **not** imply a tighter scalar-PIC completion.

This is not a contradiction: H3 has a better physical optimal value, but the particular scalar linearly-solvable relaxation can become more optimistic because its common temperature is bottlenecked by a weak-noise channel.  The natural interpretation of \(\operatorname{tr}(G_+-G)\) is therefore a **PIC mismatch / completion burden**, not actuator rank itself.

In [ ]:
for kind in ['homogeneous','activity_scaled']:
    q=df[(df.noise==kind)&(df.sigma_rms==0.1)]
    print('\n',kind)
    print(q[['H','inflation_trace','whitened_deficit_trace','gap_plus','gap_H4','bound_improvement']].to_string(index=False))

## Noise-strength sweep

For a fixed **shape** of the noise profile, scaling \(\Gamma\mapsto s\Gamma\) gives \(Q\mapsto s^2Q\) and \(\lambda_{\max}\mapsto s^2\lambda_{\max}\). Hence
\[
G_+=Q/\lambda_{\max}
\]
and the completion geometry are invariant to overall RMS noise amplitude.  The stochastic cost and certificate gaps are not invariant: they grow strongly as the noise increases.

The figures therefore separate two notions:

* completion burden diagnoses **architecture/noise-shape mismatch**;
* the value gap measures how costly that mismatch becomes at the chosen noise amplitude and objective.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(FIG/'case39_control_inflation_hierarchy.png')))
display(Image(filename=str(FIG/'case39_control_inflation_noise.png')))

## Numerical qualification

The exact completed values are mathematically valid lower bounds.  The displayed numbers are Monte-Carlo estimates of those exact Feynman--Kac values.  Importance sampling is well conditioned for homogeneous noise and for H0/H1 under activity-scaled noise, but several H2/H3 activity-scaled points have low effective sample size.  Those points are useful for trend discovery but should not yet be treated as statistically certified numerical lower bounds.

The main robust conclusions of this experiment are therefore structural:

1. H4 supplies a common curvature-free lower oracle for H0--H3.
2. Under homogeneous noise, completion burden decreases with physical rank and the H4 gap tightens.
3. Under heterogeneous noise, scalar-temperature matching creates a weakest-channel bottleneck; architecture-specific completion can become *more* inflated as rank grows.
4. The absolute completion burden tracks the architecture-specific completed gap much better than rank alone in the current activity-scaled experiment.
5. This suggests a new actuator/cost-design question: choose the physical metric \(R\) to reduce scalar matching mismatch, e.g. by approximately equalizing the channel products \(r_g\Gamma_g^2\) when independent generator controls are available.